# Uncertainty Analysis

Run after `scripts/05_uncertainty.py --split test`. Requires `results/test_uncertainty.csv`.

In [ ]:
import os, sys
from pathlib import Path

_here = Path.cwd()
while not (_here / 'pyproject.toml').exists() and _here != _here.parent:
    _here = _here.parent
os.chdir(_here)
sys.path.insert(0, str(_here / 'src'))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.stats import spearmanr

from echo_seg.uncertainty import risk_coverage_curve

RESULTS = 'results'
df = pd.read_csv(f'{RESULTS}/test_uncertainty.csv')
with open(f'{RESULTS}/test_uncertainty_summary.json') as f:
    summary = json.load(f)
print(f'CWD: {Path.cwd()}')
print(df.describe().T.round(4))

In [ ]:
# Spearman correlations: uncertainty vs. error
structures = ['lv_cavity', 'myocardium', 'left_atrium']
print('Structure           | ρ      | p-value')
print('-' * 40)
for name in structures:
    errors = 1 - df[f'dice_{name}'].values
    rho, p = spearmanr(df['uncertainty'].values, errors)
    print(f'{name:<20}| {rho:+.3f} | {p:.4f}')

In [ ]:
# Scatter: uncertainty vs. error (interactive)
fig = go.Figure()
colors = ['steelblue', 'darkorange', 'forestgreen']

for name, color in zip(structures, colors):
    errors = 1 - df[f'dice_{name}'].values
    hover = [f"{r['patient_id']} {r['view']} {r['phase']}" for _, r in df.iterrows()]
    fig.add_trace(go.Scatter(
        x=df['uncertainty'], y=errors,
        mode='markers', name=name.replace('_',' '),
        marker=dict(color=color, opacity=0.6, size=6),
        text=hover, hovertemplate='%{text}<br>unc=%{x:.4f} err=%{y:.3f}'
    ))

fig.update_layout(
    title='Uncertainty vs. Error (1-Dice) per structure',
    xaxis_title='Mean pixel entropy (uncertainty)',
    yaxis_title='Error (1 − Dice)',
    height=500
)
fig.show()

In [ ]:
# Risk-coverage curves
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, name in zip(axes, structures):
    cov = np.array(summary[name]['coverage'])
    risk = np.array(summary[name]['risk'])
    rho = summary[name]['spearman_rho']
    
    ax.plot(cov, risk, marker='o', linewidth=2)
    ax.axhline(risk[0], color='gray', linestyle='--', alpha=0.5, label='all-data baseline')
    ax.set_xlabel('Coverage (fraction retained)')
    ax.set_ylabel('Mean error (1−Dice)')
    ax.set_title(f'{name.replace("_"," ")}\nρ={rho:.3f}')
    ax.invert_xaxis()
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Risk-Coverage Curves', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Most uncertain cases
top_uncertain = df.nlargest(5, 'uncertainty')
print('Top 5 most uncertain cases:')
print(top_uncertain[['patient_id','view','phase','quality','uncertainty'] + [f'dice_{n}' for n in structures]].to_string(index=False))